# Converting my hand-drawn PDF pages into the CSV files

**What this notebook does:** it turns my 40 hand-drawn PDF pages (10 symbols x 4 pages, 25 drawings per page) into the two CSV files the neural network uses: `symbols_training.csv` and `symbols_testing.csv`.

**The idea in six steps**
1. Open each PDF page as a picture.
2. Find the 25 drawings on the page.
3. Cut each drawing out, shrink it to fit a 28 x 28 square, and save it as a PNG named `class_number.png` (for example `3_0042.png`).
4. Fix the one page that did not cut cleanly (`7_0073`).
5. Check that there are 1,000 images, 100 per class.
6. Split the images at random into 500 training and 500 test images and write them to CSV files (label first, then 784 pixel values).

**Where files go:** everything this notebook creates goes into a new folder called `conversion_output`, so it never overwrites my real CSV files. The last cell compares the result with my real CSV files.

**To run:** keep this notebook in the same folder as the 40 PDF files (named `Symbol 0.1.pdf` ... `Symbol 9.4.pdf`), then choose Kernel, then Restart and Run All Cells.

## Step 0: Imports
These are the tools used: `pymupdf` opens PDFs, `PIL` (Pillow) handles pictures, `numpy` holds pixel values as arrays, `scipy.ndimage` finds separate shapes in a picture, and `csv`, `glob`, `random`, `os` handle files and the random split.

In [ ]:
import os
import io
import csv
import glob
import random
import numpy
import pymupdf
from PIL import Image
from scipy import ndimage
import matplotlib.pyplot

## Step 1: Settings and the output folder
The PDFs are in the same folder as this notebook. The new images and CSV files are saved in `conversion_output`.

In [ ]:
output_folder = "conversion_output"
image_folder = output_folder + "/input_images"

os.makedirs(image_folder, exist_ok=True)

## Step 2: Cut the 25 drawings out of every PDF page

For each of the 40 PDF pages the loop below does the following.

1. **Open the page as a grey picture.** `get_pixmap(dpi=180)` turns the PDF page into pixels, and `.convert("L")` makes it grayscale (0 = black, 255 = white).
2. **Find the pen strokes.** Pixels darker than 190 are counted as ink.
3. **Join nearby strokes.** `binary_dilation` thickens the ink slightly so that the pieces of one drawing join up into one shape.
4. **Find the separate shapes.** `ndimage.label` gives every connected shape its own number, and `find_objects` gives the box around each shape.
5. **Keep the 25 biggest shapes.** Specks smaller than 30 pixels are ignored, and the 25 largest of the rest are the 25 drawings.
6. **Cut out each drawing** with 5 pixels of space around it.
7. **Shrink it to fit 22 x 22 pixels** with `thumbnail`. This keeps the shape's proportions, and the smooth shrinking (LANCZOS) is what gives the soft, anti-aliased grey edges.
8. **Centre it on a white 28 x 28 square** and save it.
9. **Number them in reading order**: sort into 5 rows from top to bottom, then left to right within each row. The images are named `class_number.png`, with numbers 0001 to 0100 for each class.

In [ ]:
for class_number in range(10):
    for page_number in range(1, 5):

        # 1. open the page as a grayscale picture
        pdf = pymupdf.open("Symbol " + str(class_number) + "." + str(page_number) + ".pdf")
        pixmap = pdf[0].get_pixmap(dpi=180)
        page = Image.open(io.BytesIO(pixmap.tobytes("png"))).convert("L")
        pdf.close()
        page_pixels = numpy.array(page)

        # 2-3. find the ink and join nearby strokes together
        ink = page_pixels < 190
        joined = ndimage.binary_dilation(ink, iterations=2)

        # 4. give each separate shape its own number and find its box
        shape_numbers, how_many = ndimage.label(joined)
        shape_sizes = numpy.bincount(shape_numbers.ravel())
        shape_boxes = ndimage.find_objects(shape_numbers)

        # 5. keep the 25 biggest shapes (ignore tiny specks)
        shape_sizes[0] = 0                     # number 0 is the background
        shape_sizes[shape_sizes < 30] = 0      # ignore specks
        biggest = numpy.argsort(shape_sizes)[::-1][:25]

        # 6-8. cut out each drawing and make it a 28 x 28 image
        small_images = []
        centre_y = []
        centre_x = []
        for n in biggest:
            box = shape_boxes[n - 1]
            top = max(0, box[0].start - 5)
            bottom = min(page_pixels.shape[0], box[0].stop + 5)
            left = max(0, box[1].start - 5)
            right = min(page_pixels.shape[1], box[1].stop + 5)

            drawing = Image.fromarray(page_pixels[top:bottom, left:right])
            drawing.thumbnail((22, 22), Image.Resampling.LANCZOS)

            square = Image.new("L", (28, 28), 255)
            square.paste(drawing, ((28 - drawing.width) // 2, (28 - drawing.height) // 2))

            small_images.append(square)
            centre_y.append((top + bottom) / 2)
            centre_x.append((left + right) / 2)

        # 9. put them in reading order: 5 rows from the top, left to right in each row
        centre_y = numpy.array(centre_y)
        centre_x = numpy.array(centre_x)
        top_to_bottom = numpy.argsort(centre_y)
        reading_order = []
        for row in range(5):
            row_members = top_to_bottom[row * 5:(row + 1) * 5]
            left_to_right = row_members[numpy.argsort(centre_x[row_members])]
            reading_order.extend(left_to_right)

        for position in range(25):
            image_number = (page_number - 1) * 25 + position + 1
            file_name = str(class_number) + "_" + str(image_number).zfill(4) + ".png"
            small_images[reading_order[position]].save(image_folder + "/" + file_name)

    print("Class", class_number, "done")

## Step 3: Look at one page

This is a quick visual check: the 25 images from class 3, page 2 (images 26 to 50), laid out in a 5 x 5 grid.

In [ ]:
matplotlib.pyplot.figure(figsize=(6, 6))
for i in range(25):
    img = Image.open(image_folder + "/3_" + str(26 + i).zfill(4) + ".png")
    matplotlib.pyplot.subplot(5, 5, i + 1)
    matplotlib.pyplot.imshow(img, cmap="gray", vmin=0, vmax=255)
    matplotlib.pyplot.axis("off")
matplotlib.pyplot.show()

## Step 4: Fix the one image that did not cut cleanly (`7_0073`)

On page 7.3 one symbol is drawn in two separate pieces, so the automatic step above grouped it wrongly. For this one image only, I cut it out by hand with a fixed crop box (given as fractions of the page width and height), then shrink it and centre it exactly as before. This replaces `7_0073.png`.

In [ ]:
pdf = pymupdf.open("Symbol 7.3.pdf")
pixmap = pdf[0].get_pixmap(dpi=180)
page = Image.open(io.BytesIO(pixmap.tobytes("png"))).convert("L")
pdf.close()

width, height = page.size
crop = page.crop((int(width * 0.35), int(height * 0.68), int(width * 0.49), int(height * 0.82)))

# find the ink inside the crop and cut tightly around it
crop_pixels = numpy.array(crop)
ys, xs = numpy.where(crop_pixels < 210)
symbol = crop.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))

symbol.thumbnail((20, 20), Image.Resampling.LANCZOS)
fixed = Image.new("L", (28, 28), 255)
fixed.paste(symbol, ((28 - symbol.width) // 2, (28 - symbol.height) // 2))
fixed.save(image_folder + "/7_0073.png")
print("7_0073.png replaced")

## Step 5: Check the counts
There should be exactly 1,000 images, 100 per class, each 28 x 28 pixels.

In [ ]:
all_files = sorted(glob.glob(image_folder + "/*.png"))
print("Total images:", len(all_files))
for class_number in range(10):
    files_in_class = glob.glob(image_folder + "/" + str(class_number) + "_*.png")
    print("Class", class_number, ":", len(files_in_class), "images")
for f in all_files:
    assert Image.open(f).size == (28, 28)
assert len(all_files) == 1000
print("All checks passed")

## Step 6: Split into training and test sets and write the CSV files

For each class, the 100 images are shuffled and split in half: the first 50 become training images and the other 50 become test images. This is done **before any augmentation**, so the test images are never touched afterwards.

`random.seed(42)` fixes the shuffle, so the same split is produced every time.

Each CSV row is the class label followed by 784 pixel values. The picture is inverted (`255 - pixel`) so the strokes are light on a dark background, like MNIST (this follows the book's own-handwriting example).

In [ ]:
random.seed(42)

training_rows = []
test_rows = []

for class_number in range(10):
    files_in_class = sorted(glob.glob(image_folder + "/" + str(class_number) + "_*.png"))
    random.shuffle(files_in_class)

    for file_name in files_in_class[:50]:
        pixels = 255 - numpy.array(Image.open(file_name).convert("L"))
        training_rows.append([class_number] + pixels.flatten().tolist())

    for file_name in files_in_class[50:]:
        pixels = 255 - numpy.array(Image.open(file_name).convert("L"))
        test_rows.append([class_number] + pixels.flatten().tolist())

with open(output_folder + "/symbols_training.csv", "w", newline="") as f:
    csv.writer(f).writerows(training_rows)

with open(output_folder + "/symbols_testing.csv", "w", newline="") as f:
    csv.writer(f).writerows(test_rows)

print("Training images:", len(training_rows))
print("Testing images:", len(test_rows))

## Step 7: Compare with my real CSV files
If `symbols_training.csv` and `symbols_testing.csv` are in this folder, this cell checks that the files made here are identical to them.

In [ ]:
for name in ["symbols_training.csv", "symbols_testing.csv"]:
    if os.path.exists(name):
        same = open(name).read() == open(output_folder + "/" + name).read()
        print(name, "identical to the one made here:", same)
    else:
        print(name, "not found in this folder, nothing to compare")